In [0]:
from pyspark.sql.functions import *

In [0]:
df_bronze_hospital = spark.readStream.table('healthcare.bronze.hospital')
silver_table = 'healthcare.silver.hospital'
checkpoint_path = '/Volumes/healthcare/silver/checkpoint/silver_hospital_files/v1' 

In [0]:
df_clean_hospital = df_bronze_hospital.dropDuplicates(['hospital_id']).withColumn('updated_at', current_timestamp())

In [0]:
def merge_hospital_data(batch_df,batch_id):
    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format('delta').mode('append').saveAsTable(silver_table)
        return
    
    from delta.tables import DeltaTable

    final_table = DeltaTable.forName(spark,silver_table)

    (
        final_table.alias('target')
        .merge(
            batch_df.alias('source'),
            'target.hospital_id = source.hospital_id'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


(
    df_clean_hospital
        .writeStream
        .format('delta')
        .option('checkpointLocation',checkpoint_path)
        .foreachBatch(merge_hospital_data)
        .trigger(availableNow = True)
        .start()
)